In [21]:
import json
import os
import pandas as pd

In [22]:
app_citations_to_update = []
patent_citations_to_update = []

In [64]:
def get_citation_date(engine, patent_id: str, sequence: int):
    query = sqlalchemy.text(
        f"""
                    SELECT case when p.date is not null then p.date else u.date end as date
                    FROM patent.uspatentcitation u left join patent.patent p on p.id=u.citation_id
                    WHERE patent_id = '{patent_id}' and sequence= {sequence};
    """
    )
    # query = query.bindparams(patent_id=patent_id, seq=sequence)
    df = pd.read_sql(query, engine)
    if df.shape[0] < 1:
        raise Exception
    return df.date.tolist()[0]

In [32]:
import sqlalchemy

engine = sqlalchemy.create_engine(os.environ["INGEST_ENGINE_URL"])

In [72]:
from datetime import datetime

for subdir, dirs, files in os.walk("."):
    for file in files:
        if file.endswith(".json"):
            print(file)
            content = json.load(open(file, "r"))
            for content_record in content:
                if "citation_id" in content_record:
                    cdate = get_citation_date(
                        engine=engine,
                        patent_id=content_record["patent_id"],
                        sequence=content_record["sequence"],
                    )
                    if cdate != content_record:
                        content_record["date"] = datetime.strftime(cdate, "%Y-%m-%d")
            json.dump(content, open(file, "w"))

test_eq.json
test_lte.json
test_lt.json
test_contains.json
test_s_num_seq.json
test_s_seq_asc.json
test_or.json
test_and_simple.json
test_and_nested.json
test_begins.json
test_gt.json
test_mult_records.json
test_gte.json
test_s_num_desc.json
test_no_records.json
